In [1]:
import numpy as np
import pandas as pd

In [2]:
from pathlib import Path

In [3]:
raw_movie_path= Path(r"..\data\raw\movies")

movies = pd.read_csv(raw_movie_path / "tmdb_5000_movies.csv")
credits = pd.read_csv(raw_movie_path / "tmdb_5000_credits.csv")

In [4]:
master_df = movies.merge(credits, on="title")

In [5]:
recommendation_df = master_df[['movie_id','genres', 'keywords', 'title', 'overview', 'cast', 'crew','production_companies', 'release_date', 'revenue', 'runtime']]

In [6]:
recommendation_df.dropna(inplace=True)

In [7]:
import ast

def give_genre (obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [8]:
give_genre(recommendation_df.iloc[0].genres)

['Action', 'Adventure', 'Fantasy', 'Science Fiction']

In [9]:
movies['genres'].apply(give_genre)

0       [Action, Adventure, Fantasy, Science Fiction]
1                        [Adventure, Fantasy, Action]
2                          [Action, Adventure, Crime]
3                    [Action, Crime, Drama, Thriller]
4                [Action, Adventure, Science Fiction]
                            ...                      
4798                        [Action, Crime, Thriller]
4799                                [Comedy, Romance]
4800               [Comedy, Drama, Romance, TV Movie]
4801                                               []
4802                                    [Documentary]
Name: genres, Length: 4803, dtype: object

In [10]:
recommendation_df['genres'] = recommendation_df['genres'].apply(give_genre)

In [11]:
recommendation_df['keywords'] = recommendation_df['keywords'].apply(give_genre)

In [12]:
recommendation_df.head(2)

,movie_id,genres,keywords,title,overview,cast,crew,production_companies,release_date,revenue,runtime
0,19995,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...",Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...","[{""name"": ""Ingenious Film Partners"", ""id"": 289...",2009-12-10,2787965087,162.0
1,285,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...",Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...","[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",2007-05-19,961000000,169.0


In [13]:
def give_cast_name (obj):
    L = []
    for i in ast.literal_eval(obj):
        if(len(L) == 3):
            break
        L.append(i['name'])

    return L

In [14]:
recommendation_df['cast'] = recommendation_df['cast'].apply(give_cast_name)

In [15]:
def give_director_name (obj):
    L = []
    for i in ast.literal_eval(obj):
        if( i['job'] == 'Director'):
            L.append(i['name'])
            break
    return L

In [16]:
recommendation_df['crew'] = recommendation_df['crew'].apply(give_director_name)

In [17]:
recommendation_df.rename(columns={'crew':'director'},inplace=True)

In [18]:
recommendation_df['overview'] = recommendation_df['overview'].apply(lambda x: x.split())

In [19]:
def removespace(obj):
    L=[]
    for i in obj:
        L.append(i.replace(" ",""))

    return L

In [20]:
recommendation_df['genres'] = recommendation_df['genres'].apply(removespace)
recommendation_df['keywords'] = recommendation_df['keywords'].apply(removespace)
recommendation_df['cast'] = recommendation_df['cast'].apply(removespace)
recommendation_df['director'] = recommendation_df['director'].apply(removespace)

In [21]:
recommendation_df['tag'] = recommendation_df['genres'] + recommendation_df['keywords'] + recommendation_df['overview'] + recommendation_df['cast'] + recommendation_df['director']

In [22]:
recommendation_df['tag'] = recommendation_df['tag'].apply(lambda x: " ".join(x))

In [23]:
recommendation_df['tag'] = recommendation_df['tag'].apply(lambda x: x.lower())

In [25]:
movies.head(2)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [26]:
from pathlib import Path

processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

recommendation_df.to_pickle(processed_path/"movies1234.pkl")